# Huấn luyện CNN từ ảnh và kiểm tra bằng video

Notebook này không dùng LSTM. Mô hình phân loại từng ảnh; khi kiểm tra video, lấy mẫu frame và làm mượt dự đoán. Dataset cần có thư mục class-name/images hoặc cấu trúc `train/val/test/class-name/images`.

**Lưu ý nhãn:** model lưu class theo tên thư mục của dataset. Bộ 5 lớp Kaggle không tương đương bộ nhãn `normal/drowsy/sleepy/drinking/phone` của pipeline cũ. Chỉ so sánh với ground truth nếu đã xây mapping đúng. Nếu ảnh cùng video/người lái, hãy tự chia theo video/người lái trước để tránh leakage; phần tự chia ngẫu nhiên bên dưới chỉ dùng khi dataset ảnh không có thông tin nhóm.

Trên Kaggle bật GPU và gắn dataset. Trên Colab đổi `PLATFORM` rồi mount Drive hoặc đặt `IMAGE_SOURCE` thủ công.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
PLATFORM = 'kaggle'  # 'colab' hoặc 'local'
if PLATFORM == 'kaggle':
    REPO = Path('/kaggle/working/Driver_behaviour')
    INPUT_ROOT = Path('/kaggle/input')
elif PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/Driver_behaviour')
    INPUT_ROOT = Path('/content/drive/MyDrive')
else:
    REPO = Path.cwd()
    INPUT_ROOT = REPO
REPO_URL = 'https://github.com/tunaleeZ/Driver_behaviour.git'
if not (REPO / 'src/dbm/train_images.py').exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-e','.','--no-deps','-q'],check=True)


## 1. Chọn dataset ảnh

`IMAGE_SOURCE` phải trỏ đến thư mục có mỗi lớp thành một thư mục con, ví dụ `.../images/safe driving/*.jpg`. Nếu dataset đã chia tập sẵn thì chọn root có `train/`, `val/`, `test/`; nếu chỉ có thư mục lớp, notebook chia ngẫu nhiên 70/15/15 theo ảnh.

In [ ]:
# Sửa đường dẫn cho dataset của bạn. Kaggle: xem danh sách tự dò phía dưới.
IMAGE_SOURCE = None  # ví dụ Path('/kaggle/input/my-driver-images/images')
if IMAGE_SOURCE is None and PLATFORM == 'kaggle':
    candidates=[]
    for p in INPUT_ROOT.rglob('*'):
        if p.is_dir() and (p/'train').is_dir() and any((p/'train').iterdir()): candidates.append(p)
    # Nếu dataset chỉ gồm class folders, đặt IMAGE_SOURCE chính xác ở đây.
    print('Dataset có sẵn split:', candidates[:20])
    print('Kaggle input roots:', [str(p) for p in INPUT_ROOT.iterdir()])


In [ ]:
IMAGE_SOURCE = Path(IMAGE_SOURCE) if IMAGE_SOURCE else None
assert IMAGE_SOURCE is not None and IMAGE_SOURCE.exists(), 'Hãy gán IMAGE_SOURCE đến thư mục dataset ảnh.'
DATA_ROOT = Path('/kaggle/working/driver_images') if PLATFORM == 'kaggle' else Path('/content/driver_images')
if all((IMAGE_SOURCE/s).is_dir() for s in ('train','val','test')):
    DATA_ROOT = IMAGE_SOURCE
else:
    subprocess.run([sys.executable,'-m','dbm.prepare_image_dataset','--source',str(IMAGE_SOURCE),'--output',str(DATA_ROOT)],check=True)
for split in ('train','val','test'):
    print(split, {p.name:len(list(p.glob('*'))) for p in (DATA_ROOT/split).iterdir() if p.is_dir()})


## 2. Train CNN image classifier

Huấn luyện dùng class weighting và augmentation. Kết quả lưu checkpoint tốt nhất theo macro-F1 validation cùng metrics test. Không dùng tập test để chọn epoch.

In [ ]:
RUN_DIR = Path('/kaggle/working/outputs/cnn_image') if PLATFORM == 'kaggle' else Path('/content/outputs/cnn_image')
subprocess.run([sys.executable,'-m','dbm.train_images','--data',str(DATA_ROOT),'--output',str(RUN_DIR),'--epochs','20','--batch-size','32','--size','224'],check=True)
print(json.loads((RUN_DIR/'results.json').read_text()))


## 3. Suy luận trên video

Đặt video cần kiểm tra. Output CSV có nhãn và xác suất ở từng frame được lấy mẫu (mặc định 5 FPS). Model có thể chạy gần realtime tùy GPU/CPU, độ phân giải và FPS lấy mẫu; notebook này không tự đảm bảo throughput realtime.

In [ ]:
VIDEO_PATH = None  # ví dụ Path('/kaggle/input/my-video/test.mp4')
if VIDEO_PATH:
    VIDEO_PATH=Path(VIDEO_PATH)
    PREDICTIONS=RUN_DIR/'video_predictions.csv'
    subprocess.run([sys.executable,'-m','dbm.infer_video','--video',str(VIDEO_PATH),'--checkpoint',str(RUN_DIR/'best.pt'),'--output',str(PREDICTIONS),'--smoothing','ema','--span','5'],check=True)
    print(PREDICTIONS)
